<div align="center">
<h1>🔐 KRİPTOQRAFİYA KURSU</h1>
<h2>Məşğələ 19</h2>
<h2>RSA Kriptosistemi</h2>
<h3 style="color: #8B4513;">Rivest-Shamir-Adleman - asimmetrik kriptoqrafiyanın qabaqcılı</h3>
<br>
<h3>Məşğələ vaxtı: 2 saat</h3>
<br>
<p><em>Hazırlanma tarixi: 2024</em></p>
</div>

## 📑 Mündəricat

1. [Məşğələnin məqsədləri](#məşğələnin-məqsədləri)
2. [Lazım olan kitabxanalar](#lazım-olan-kitabxanalar)
3. [Hazırlıq (15 dəq)](#hazırlıq-15-dəq)
4. [Xatırlatma: Asimmetrik kriptoqrafiyanın əsasları (5 dəq)](#xatırlatma-asimmetrik-kriptoqrafiyanın-əsasları-5-dəq)
5. [RSA açar generasiyası (20 dəq)](#rsa-açar-generasiyası-20-dəq)
6. [RSA şifrləmə və deşifrləmə (15 dəq)](#rsa-şifrləmə-və-deşifrləmə-15-dəq)
7. [Tədris üçün kiçik RSA nümunəsi (10 dəq)](#tədris-üçün-kiçik-rsa-nümunəsi-10-dəq)
8. [Kiçik eksponent hücumları (15 dəq)](#kiçik-eksponent-hücumları-15-dəq)
9. [RSA kodlaşdırma sxemləri (15 dəq)](#rsa-kodlaşdırma-sxemləri-şifrləmə-və-imza-15-dəq)
10. [PyCryptodome ilə tam RSA tətbiqi (15 dəq)](#pycryptodome-ilə-tam-rsa-tətbiqi-15-dəq)
11. [İnteqrasiya edilmiş özünü-yoxlama (15 dəq)](#inteqrasiya-edilmiş-özünü-yoxlama-15-dəq)
12. [Ev tapşırığı](#ev-tapşırığı)
13. [Yekun və müzakirə sualları](#yekun-və-müzakirə-sualları)
14. [Əlavə resurslar](#əlavə-resurslar)

## 🎯 Məşğələnin məqsədləri

Bu məşğələni bitirdikdən sonra siz:

✅ RSA kriptosisteminin iş prinsipini izah edə biləcəksiniz  
✅ RSA açar generasiyasını Python-da implementasiya edə biləcəksiniz  
✅ RSA şifrləmə və deşifrləmə əməliyyatlarını yerinə yetirə biləcəksiniz  
✅ Kiçik eksponent hücumlarını (e=3) başa düşəcəksiniz  
✅ RSA-OAEP doldurma sxemini izah edə biləcəksiniz  
✅ RSA-PSS imza sxemini başa düşəcəksiniz  
✅ PyCryptodome kitabxanası ilə RSA istifadə edə biləcəksiniz

## 📚 Lazım olan kitabxanalar

| Kitabxana | İstifadə sahəsi |
|-----------|-----------------|
| Python standard library | Riyazi funksiyalar, təhlükəsiz təsadüfi ədədlər, vaxt ölçümü |
| PyCryptodome | RSA açar generasiyası, RSAES-OAEP, RSASSA-PSS və hibrid şifrləmə |

> 💡 **Mühit müqaviləsi:** notebook şəbəkədən paket quraşdırmır. `pycryptodome` cari
> kernel mühitində əvvəlcədən quraşdırılmalıdır; import alınmasa, ilk code cell aydın
> xəta ilə dayanacaq. `cryptography` və `gmpy2` bu praktikada istifadə edilmir.

In [ ]:
# Asılılığı cari kernel-də yoxlayırıq; notebook daxilindən şəbəkə quraşdırması etmirik.
import importlib.util
import sys

required_modules = {"Crypto": "pycryptodome"}
missing = [package for module, package in required_modules.items()
           if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError(
        "Çatışmayan paket(lər): " + ", ".join(missing)
        + ". Notebook-u işə salmazdan əvvəl kernel mühitində quraşdırın."
    )

print(f"✅ Mühit hazırdır: Python {sys.version.split()[0]}, pycryptodome mövcuddur")

## 🔧 Hazırlıq (15 dəq)

### 3.1 Python mühitinin yoxlanılması

Aşağıdakı kodu işə salaraq lazımi modulların yükləndiyini yoxlayın:

In [ ]:
import math
import secrets
import time
from itertools import combinations
from pathlib import Path

from Crypto.Cipher import AES, PKCS1_OAEP
from Crypto.Hash import SHA256
from Crypto.PublicKey import RSA
from Crypto.Random import get_random_bytes
from Crypto.Signature import pss

CRYPTO_AVAILABLE = True
print("✅ PyCryptodome import edildi: RSAES-OAEP, RSASSA-PSS və AES-GCM hazırdır")

### 3.2 İşçi qovluğun yaradılması

In [ ]:
# Notebook cari qovluğu dəyişmir və mənbə ağacında fayl yaratmır.
workspace = Path.cwd()
print(f"📁 İşçi qovluq: {workspace}")

## 📖 Xatırlatma: Asimmetrik kriptoqrafiyanın əsasları (5 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔑 Açıq və gizli açarın rolu sxemdən asılıdır</h4>
<ul>
    <li><b>Şifrləmə sxemi:</b> alıcının açıq açarı ilə şifrləmə, gizli açarı ilə deşifrləmə.</li>
    <li><b>İmza sxemi:</b> imzalayanın gizli açarı ilə imza yaratma, açıq açarı ilə yoxlama.</li>
    <li>Şifrləmə və imza ayrıca RSAES/RSASSA sxemləridir; “private key ilə şifrləmə” eyni anlayış deyil.</li>
</ul>
</div>

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4; margin-top: 10px;">
<h4>🧮 RSA-nın riyazi modeli</h4>
<ul>
    <li><b>Modul:</b> $n=pq$, burada $p$ və $q$ fərqli böyük sadə ədədlərdir.</li>
    <li><b>İnvariant:</b> $ed\equiv1\pmod{\lambda(n)}$ (və ya tədris variantında $\phi(n)$).</li>
    <li><b>Təhlükəsizlik:</b> yalnız faktorizasiya şüarından deyil, seçilmiş RSAES/RSASSA encoding-dən, parametr generasiyasından və implementasiyadan asılıdır.</li>
</ul>
</div>

## 🔑 RSA açar generasiyası (20 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔑 RSA açar generasiyası</h4>
<ol>
    <li>Fərqli böyük sadə $p$ və $q$ seçin.</li>
    <li>$n=pq$, $\phi(n)=(p-1)(q-1)$ və ya $\lambda(n)=\mathrm{lcm}(p-1,q-1)$ hesablayın.</li>
    <li>$1&lt;e&lt;\lambda(n)$ və $\gcd(e,\lambda(n))=1$ seçin.</li>
    <li>$d=e^{-1}\bmod\lambda(n)$ hesablayın.</li>
    <li>Açıq açar $(n,e)$-dir; real private key yalnız $(n,d)$ qısaltması deyil, adətən $p,q$ və CRT komponentlərini də saxlayır.</li>
    <li>Production açarını əl ilə yazılmış bu generatorla deyil, yoxlanılmış kitabxana ilə yaradın.</li>
</ol>
</div>

In [ ]:
def egcd(a, b):
    """Genişləndirilmiş Evklid alqoritmi"""
    if a == 0:
        return (b, 0, 1)
    else:
        g, x1, y1 = egcd(b % a, a)
        x = y1 - (b // a) * x1
        y = x1
        return (g, x, y)

def modinv(a, m):
    """a^-1 mod m"""
    g, x, y = egcd(a, m)
    if g != 1:
        raise ValueError(f"{a} ədədinin {m} moduluna görə tərsi yoxdur")
    return x % m

In [ ]:
_secure_random = secrets.SystemRandom()

def is_prime(n, rounds=32):
    """Tədris üçün ehtimallı Miller–Rabin testi."""
    if n < 2:
        return False
    small_primes = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37)
    for prime in small_primes:
        if n % prime == 0:
            return n == prime

    d = n - 1
    s = 0
    while d % 2 == 0:
        d //= 2
        s += 1

    for _ in range(rounds):
        a = _secure_random.randrange(2, n - 1)
        x = pow(a, d, n)
        if x in (1, n - 1):
            continue
        for _ in range(s - 1):
            x = (x * x) % n
            if x == n - 1:
                break
        else:
            return False
    return True

def generate_prime(bits):
    """Verilmiş bit ölçüsündə ehtimallı sadə ədəd generasiya edir."""
    if bits < 16:
        raise ValueError("bits ən azı 16 olmalıdır")
    while True:
        candidate = secrets.randbits(bits)
        candidate |= (1 << (bits - 1)) | 1
        if is_prime(candidate):
            return candidate

In [ ]:
def rsa_keygen(bits=256):
    """Yalnız tədris üçün textbook RSA açar generasiyası."""
    if bits < 64:
        raise ValueError("tədris nümunəsində bits ən azı 64 olmalıdır")

    p_bits = bits // 2
    q_bits = bits - p_bits
    e = 65537

    while True:
        p = generate_prime(p_bits)
        q = generate_prime(q_bits)
        if p != q and math.gcd(e, (p - 1) * (q - 1)) == 1:
            break

    n = p * q
    phi = (p - 1) * (q - 1)
    d = modinv(e, phi)

    assert p != q
    assert math.gcd(e, phi) == 1
    assert (e * d) % phi == 1
    for message in (0, 1, 2, 42, n - 1):
        assert pow(pow(message, e, n), d, n) == message

    print(f"✅ Tədris açarı: n={n.bit_length()} bit, e={e}; invariantlər keçdi")
    return (e, n), (d, n), p, q

# Kiçik ölçü yalnız alqoritmi və invariantləri görmək üçündür, real təhlükəsizlik üçün deyil.
(e, n), (d, n2), p, q = rsa_keygen(256)
assert n == n2

### 5.1 PyCryptodome ilə RSA açar generasiyası

In [ ]:
def pycryptodome_rsa_keygen_demo(bits=2048):
    """PyCryptodome ilə standart RSA açar generasiyası."""
    if bits < 1024 or bits % 256 != 0:
        raise ValueError("PyCryptodome üçün bits >= 1024 və 256-nın qatı olmalıdır")

    start = time.perf_counter()
    key = RSA.generate(bits)
    elapsed_ms = (time.perf_counter() - start) * 1000

    assert key.has_private()
    assert key.n.bit_length() == bits
    assert key.e == 65537
    assert key.p != key.q
    print(f"✅ {bits}-bit RSA açarı yaradıldı: e={key.e}, vaxt={elapsed_ms:.2f} ms")
    print("🔐 Gizli komponentlər və private PEM ekrana çıxarılmır")
    return key

key = pycryptodome_rsa_keygen_demo()

### ✍️ Çalışma 1: RSA açar generasiyası (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Əl ilə hesablama:** $p=61$, $q=53$ üçün $n$, $\phi(n)$, $e=17$ üçün $d$-ni hesablayın.

2. **Tədris benchmark-ı:** yalnız vaxt ölçmək üçün 1024-bit RSA açarı yaradın və ölçməni bu maşına aid edin. 1024 biti production təhlükəsizlik tövsiyəsi kimi təqdim etməyin.

3. **$e$ seçimi:** hansı qarşılıqlı sadəlik şərti tələb olunur və $e=65537$ niyə geniş istifadə edilir?

4. **Səhv vəziyyəti:** $\gcd(e,\lambda(n))\ne1$ olarsa nə baş verir?

In [ ]:
# Çalışma 1 - yoxlanılan cavablar

p1, q1, e1 = 61, 53, 17
n1 = p1 * q1
phi1 = (p1 - 1) * (q1 - 1)
d1 = modinv(e1, phi1)
assert (n1, phi1, d1) == (3233, 3120, 2753)
assert (e1 * d1) % phi1 == 1
print(f"1. n={n1}, φ(n)={phi1}, d={d1} — yoxlama keçdi")

start = time.perf_counter()
exercise_key = RSA.generate(1024)
elapsed_ms = (time.perf_counter() - start) * 1000
assert exercise_key.n.bit_length() == 1024
print(f"2. Faktiki 1024-bit tədris generasiyası: {elapsed_ms:.2f} ms (bu maşında)")

print("""
3. e seçimi:
   • 1 < e < λ(n) və gcd(e, λ(n)) = 1 olmalıdır;
   • 65537 geniş yayılmış seçimdir: təkdir, sadədir və ikilik çəkisi kiçikdir;
   • kiçik e özü avtomatik sındırılma demək deyil, lakin textbook RSA ilə e=3
     birbaşa kök və broadcast hücumlarını mümkün edir. Standart sxem/API istifadə edin.

4. gcd(e, φ(n)) ≠ 1 olduqda modul tərs d mövcud deyil; belə parametr RSA açar cütü deyil.
""")

## 🔐 RSA şifrləmə və deşifrləmə (15 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔐 Textbook RSA primitive-i</h4>
<ul>
    <li><b>Domen:</b> $m\in\{0,\ldots,n-1\}$.</li>
    <li><b>Public əməliyyat:</b> $c=m^e\bmod n$.</li>
    <li><b>Private əməliyyat:</b> $m=c^d\bmod n$.</li>
    <li>Bu düsturlar təkbaşına real şifrləmə sxemi deyil; real API RSAES-OAEP kimi encoding tətbiq edir.</li>
</ul>
</div>

In [ ]:
def textbook_rsa_roundtrip(message, p=61, q=53, e=17):
    """Yalnız 0 <= message < n tam ədədi üçün textbook RSA nümayişi."""
    if p == q or not is_prime(p) or not is_prime(q):
        raise ValueError("p və q fərqli sadə ədədlər olmalıdır")
    n = p * q
    phi = (p - 1) * (q - 1)
    if math.gcd(e, phi) != 1:
        raise ValueError("gcd(e, φ(n)) = 1 olmalıdır")
    if not 0 <= message < n:
        raise ValueError("textbook RSA üçün 0 <= m < n olmalıdır")

    d = modinv(e, phi)
    ciphertext = pow(message, e, n)
    recovered = pow(ciphertext, d, n)
    assert recovered == message
    return ciphertext, recovered

c_demo, m_demo = textbook_rsa_roundtrip(42)
print(f"✅ Textbook round-trip: m=42, c={c_demo}, bərpa={m_demo}")

In [ ]:
def textbook_byte_blocks_demo(message):
    """Baytları ayrıca RSA blokları kimi göstərən, təhlükəsiz olmayan tədris codec-i."""
    data = message.encode("utf-8")
    p, q, e = 61, 53, 17
    n = p * q
    if n <= 255:
        raise ValueError("hər baytın n-dən kiçik olması tələb olunur")
    phi = (p - 1) * (q - 1)
    d = modinv(e, phi)
    encrypted = [pow(byte, e, n) for byte in data]
    recovered_bytes = bytes(pow(value, d, n) for value in encrypted)
    recovered = recovered_bytes.decode("utf-8")
    assert recovered == message
    return encrypted, recovered

byte_ciphertext, byte_plaintext = textbook_byte_blocks_demo("Salam, RSA!")
print(f"✅ Bayt-blok round-trip: {byte_plaintext!r}, blok sayı={len(byte_ciphertext)}")
print("⚠️ Bu codec təhlükəsiz deyil: patternləri gizlətmir. Real uzun məlumat üçün hibrid AEAD istifadə olunur.")

### ✍️ Çalışma 2: RSA şifrləmə (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. $p=17$, $q=19$, $e=5$ üçün $m=123$ mesajını şifrləyin və deşifrləyin.

2. $n=143$, $e=7$, $c=48$ verilib. Deşifrləmə üçün $d$-ni tapın və $m$-i hesablayın.

3. **Nəzəri sual:** Niyə $m < n$ olmalıdır? Əgər $m \geq n$ olarsa nə etmək olar?

In [ ]:
# Çalışma 2 - yoxlanılan cavablar

p1, q1, e1, m1 = 17, 19, 5, 123
n1 = p1 * q1
phi1 = (p1 - 1) * (q1 - 1)
d1 = modinv(e1, phi1)
c1 = pow(m1, e1, n1)
m1_dec = pow(c1, d1, n1)
assert 0 <= m1 < n1 and m1_dec == m1
print(f"1. n={n1}, d={d1}, c={c1}, bərpa={m1_dec}")

n2, e2, c2 = 143, 7, 48
p2, q2 = 11, 13
phi2 = (p2 - 1) * (q2 - 1)
d2 = modinv(e2, phi2)
m2 = pow(c2, d2, n2)
assert pow(m2, e2, n2) == c2
print(f"2. φ(n)={phi2}, d={d2}, m={m2}; yenidən şifrləmə yoxlaması keçdi")

print("""
3. Textbook RSA Z_n üzərində işləyir: m >= n olduqda yalnız m mod n saxlanır və ilkin
   tam ədəd itir. Mətn üçün özbaşına böyük bloklar yaratmaq olmaz. Real sistemdə uzun
   məlumat AEAD ilə, təsadüfi AEAD açarı isə RSAES-OAEP ilə şifrlənir.
""")

## 📚 Tədris üçün kiçik RSA nümunəsi (10 dəq)

<div style="background-color: #fff3e0; padding: 15px; border-radius: 10px; border-left: 5px solid #ff9800;">
<h4>📜 Parametrlərin mənşəyi</h4>
<p><code>p=61, q=53, e=17, d=2753</code> kombinasiyası geniş yayılmış kiçik tədris nümunəsidir; onu RSA müəlliflərinin 1978-ci il məqaləsindəki konkret mətn nümunəsi kimi təqdim etmirik. Kiçik modul yalnız hesabı əl ilə izləmək üçündür.</p>
<ul>
    <li>$n = 3233$, $\phi(n) = 3120$</li>
    <li>$e = 17$, $d = 2753$</li>
</ul>
</div>

In [ ]:
def small_rsa_text_example(text="RSA"):
    """Hər UTF-8 baytını ayrıca kiçik textbook RSA blokunda göstərir."""
    encrypted, recovered = textbook_byte_blocks_demo(text)
    assert recovered == text
    print(f"📨 Mətn: {text!r}")
    print(f"🔒 Tədris blokları: {encrypted}")
    print(f"🔓 Bərpa: {recovered!r}")
    return encrypted, recovered

classic_blocks, classic_recovered = small_rsa_text_example()

## ⚠️ Kiçik eksponent hücumları (15 dəq)

<div style="background-color: #ffebee; padding: 15px; border-radius: 10px; border-left: 5px solid #f44336;">
<h4>⚠️ Kiçik eksponent hücumu (e=3)</h4>
<p>Əgər $e=3$ və mesaj kiçikdirsə ($m^3 < n$), onda şifrəli mətnin kub kökünü tapmaqla mesaj bərpa oluna bilər.</p>
</div>

In [ ]:
def integer_nth_root_exact(value, degree):
    """Mənfi olmayan value üçün tam degree-ci kökü və exact flag-i qaytarır."""
    if value < 0 or degree < 1:
        raise ValueError("value >= 0 və degree >= 1 olmalıdır")
    low, high = 0, 1
    while high ** degree < value:
        high *= 2
    while low + 1 < high:
        mid = (low + high) // 2
        if mid ** degree <= value:
            low = mid
        else:
            high = mid
    root = high if high ** degree == value else low
    return root, root ** degree == value

def small_exponent_attack_demo():
    """Textbook RSA-da m^3 < n olduqda birbaşa tam kök hücumu."""
    p, q, e, message = 101, 113, 3, 20
    n = p * q
    phi = (p - 1) * (q - 1)
    assert math.gcd(e, phi) == 1
    assert message ** e < n
    ciphertext = pow(message, e, n)
    recovered, exact = integer_nth_root_exact(ciphertext, e)
    assert exact and recovered == message
    print(f"✅ Birbaşa kök: n={n}, c={ciphertext}, m={recovered}")

    wrapped_message = 42
    wrapped_ciphertext = pow(wrapped_message, e, n)
    candidate, exact = integer_nth_root_exact(wrapped_ciphertext, e)
    assert not (exact and candidate == wrapped_message)
    print("ℹ️ m^3 >= n olduqda tək ciphertext-in adi tam kub kökü mesajı vermir.")
    return recovered

assert small_exponent_attack_demo() == 20

### 8.1 Håstad yayım hücumu

<div style="background-color: #ffebee; padding: 15px; border-radius: 10px; border-left: 5px solid #f44336;">
<h4>⚠️ Håstad yayım hücumu</h4>
<p>Eyni <b>kodlaşdırılmamış</b> mesaj eyni kiçik $e$ ilə ən azı $e$ alıcının cüt-cüt qarşılıqlı sadə RSA modulları altında şifrlənərsə, CRT $m^e$ tam ədədini bərpa edə bilər. Randomized RSAES-OAEP eyni mesajı bu formada təkrar etmir.</p>
</div>

In [ ]:
def crt_pairwise(residues, moduli):
    if len(residues) != len(moduli) or not residues:
        raise ValueError("eyni sayda boş olmayan residue/modulus siyahısı tələb olunur")
    if any(math.gcd(a, b) != 1 for a, b in combinations(moduli, 2)):
        raise ValueError("modullar cüt-cüt qarşılıqlı sadə olmalıdır")
    product = math.prod(moduli)
    value = 0
    for residue, modulus in zip(residues, moduli):
        partial = product // modulus
        value += residue * partial * modinv(partial, modulus)
    return value % product, product

def hastad_broadcast_attack_demo():
    """e=3 üçün üç valid və pairwise-coprime RSA modulu ilə Håstad demosu."""
    prime_pairs = ((101, 113), (107, 131), (137, 149))
    e, message = 3, 42
    moduli = [p * q for p, q in prime_pairs]
    for p, q in prime_pairs:
        assert math.gcd(e, (p - 1) * (q - 1)) == 1
    assert all(math.gcd(a, b) == 1 for a, b in combinations(moduli, 2))

    ciphertexts = [pow(message, e, modulus) for modulus in moduli]
    reconstructed, product = crt_pairwise(ciphertexts, moduli)
    assert message ** e < product
    assert reconstructed == message ** e
    recovered, exact = integer_nth_root_exact(reconstructed, e)
    assert exact and recovered == message
    print(f"✅ Håstad: modullar={moduli}, CRT={reconstructed}, bərpa={recovered}")
    return recovered

assert hastad_broadcast_attack_demo() == 42

### 8.2 Wiener hücumu (kiçik $d$)

<div style="background-color: #ffebee; padding: 15px; border-radius: 10px; border-left: 5px solid #f44336;">
<h4>⚠️ Wiener hücumu</h4>
<p>Wiener davamlı kəsrlərlə müəyyən balanslı iki-sadəli RSA açarlarında kifayət qədər kiçik $d$-ni tapır. Tez-tez verilən $d&lt;n^{1/4}/3$ həddi uyğun əlavə fərziyyələr altında kafi şərtdir; bütün RSA açarları üçün “əgər və yalnız əgər” meyarı deyil.</p>
</div>

### ✍️ Çalışma 3: Kiçik eksponent hücumları (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. $n=11413$, $e=3$, $c=8000$ verilib və $c=m^3<n$ olduğu məlumdur. Mesajı tam ədəd kub kökü ilə tapın.

2. $e=3$ hansı məhz textbook RSA şərtlərində birbaşa kök və Håstad hücumlarına yol açır? RSAES-OAEP bu modeli necə dəyişir?

3. Kiçik $d$ seçmək niyə təhlükəlidir? Wiener hücumunun fərziyyələri hansılardır?

4. Praktik açar generasiyasında niyə $d$-ni əl ilə “təsadüfi” seçmək əvəzinə standart kitabxanaya həvalə edirik?

In [ ]:
# Çalışma 3 - yoxlanılan cavablar

n, e, ciphertext = 11413, 3, 8000
message, exact = integer_nth_root_exact(ciphertext, e)
assert exact and message == 20
assert pow(message, e, n) == ciphertext
print(f"1. m=∛{ciphertext}={message}; yenidən şifrləmə yoxlaması keçdi")

print("""
2. Risk kiçik e-nin tək mövcudluğu deyil, randomized encoding olmadan textbook RSA-dır:
   m^e < n olduqda birbaşa kök, e alıcıya eyni m göndərildikdə Håstad işləyə bilər.
   RSAES-OAEP hər şifrləməni randomlaşdırır və mesajı standart encoding ilə xəritələyir.

3. Wiener hücumu balanslı p,q və kifayət qədər kiçik d kimi fərziyyələr altında e/n-in
   davamlı kəsrlərindən namizəd d-lər çıxarır. Sadə n^(1/4)/3 ifadəsi universal iff deyil.

4. e=65537 geniş yayılmış public exponent-dir. d ayrıca "n-in yarısı" və ya təsadüfi
   seçilmir: valid p,q,e-dən modul tərs kimi hesablanır və standart generator əlavə
   ölçü/struktur yoxlamalarını tətbiq edir.
""")

## 📦 RSA kodlaşdırma sxemləri: şifrləmə və imza (15 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>📌 Sxemləri ayırın</h4>
<ul>
    <li><b>Şifrləmə:</b> RSAES-OAEP və legacy RSAES-PKCS1-v1_5.</li>
    <li><b>İmza:</b> RSASSA-PSS və RSASSA-PKCS1-v1_5.</li>
    <li>OAEP-in şifrələmə təhlükəsizlik terminlərini PSS imzasına tətbiq etmək olmaz.</li>
    <li>Textbook RSA real mesaj və ya imza formatı deyil.</li>
</ul>
</div>

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4; margin-top: 10px;">
<h4>🔐 RSAES-OAEP</h4>
<p>RFC 8017-də RSAES-OAEP şifrləmə sxemidir: hash/MGF, optional label və təsadüfi seed ilə encoding yaradır. Mesaj ölçüsü modul və hash-dən asılı olaraq məhduddur; uzun məlumat üçün aşağıdakı hibrid RSA+AEAD nümunəsinə baxın.</p>
</div>

In [ ]:
def rsa_oaep_demo():
    """PyCryptodome ilə RSAES-OAEP round-trip və mənfi tamper testi."""
    key = RSA.generate(2048)
    message = "Bu gizli mesajdır.".encode("utf-8")
    ciphertext = PKCS1_OAEP.new(key.publickey(), hashAlgo=SHA256).encrypt(message)
    decrypted = PKCS1_OAEP.new(key, hashAlgo=SHA256).decrypt(ciphertext)
    assert decrypted == message

    tampered = bytearray(ciphertext)
    tampered[-1] ^= 1
    try:
        PKCS1_OAEP.new(key, hashAlgo=SHA256).decrypt(bytes(tampered))
    except ValueError:
        pass
    else:
        raise AssertionError("dəyişdirilmiş OAEP ciphertext rədd edilmədi")

    print(f"✅ RSAES-OAEP: {len(message)} bayt -> {len(ciphertext)} bayt; tamper rədd edildi")
    return key, ciphertext, decrypted

oaep_key, oaep_ciphertext, oaep_plaintext = rsa_oaep_demo()

### 9.1 RSASSA-PSS (imza sxemi)

In [ ]:
def rsa_pss_demo():
    """RSASSA-PSS imza, verification və yanlış-mesaj testi."""
    key = RSA.generate(2048)
    message = "Bu vacib sənəddir.".encode("utf-8")
    signature = pss.new(key).sign(SHA256.new(message))
    pss.new(key.publickey()).verify(SHA256.new(message), signature)

    try:
        pss.new(key.publickey()).verify(SHA256.new(message + b"!"), signature)
    except (ValueError, TypeError):
        pass
    else:
        raise AssertionError("imza dəyişdirilmiş mesaj üçün qəbul edildi")

    print(f"✅ RSASSA-PSS: {len(signature)} bayt imza; yanlış mesaj rədd edildi")
    return key, signature

pss_key, pss_signature = rsa_pss_demo()

### 9.2 PKCS #1 v2.2 sxemlərinin düzgün təsnifatı

In [ ]:
def pkcs_comparison():
    rows = (
        ("RSAES-OAEP", "şifrləmə", "bəli", "yeni dizaynda protokol profili ilə"),
        ("RSAES-PKCS1-v1_5", "şifrləmə", "bəli", "yalnız legacy uyğunluq; oracle müdafiəsi vacibdir"),
        ("RSASSA-PSS", "imza", "bəli (salt)", "yeni imzalar üçün müasir seçim"),
        ("RSASSA-PKCS1-v1_5", "imza", "xeyr", "legacy/interoperability kontekstində"),
    )
    print("Sxem | Əməliyyat | Randomized | Qeyd")
    print("-" * 96)
    for row in rows:
        print(" | ".join(row))
    assert {row[1] for row in rows} == {"şifrləmə", "imza"}
    return rows

pkcs_rows = pkcs_comparison()

### ✍️ Çalışma 4: RSAES və RSASSA sxemləri (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **OAEP testi:** RSAES-OAEP ilə bir mesajı şifrləyin, deşifrləyin və dəyişdirilmiş ciphertext-in rəddini yoxlayın.

2. **PSS testi:** RSASSA-PSS ilə bir mesajı imzalayın, düzgün və dəyişdirilmiş mesajları yoxlayın.

3. **Bleichenbacher hücumu:** RSAES-PKCS1-v1_5 padding oracle hədəf plaintext-i necə daraldır? Niyə bu, private key-in çıxarılması demək deyil?

4. **Təsnifat:** RSAES-PKCS1-v1_5, RSAES-OAEP, RSASSA-PKCS1-v1_5 və RSASSA-PSS-i əməliyyat və randomization üzrə ayırın.

In [ ]:
# Çalışma 4 - cavabların əsas invariantləri yuxarıdakı iki real testdə yoxlanıldı.

assert oaep_plaintext == "Bu gizli mesajdır.".encode("utf-8")
pss.new(pss_key.publickey()).verify(
    SHA256.new("Bu vacib sənəddir.".encode("utf-8")), pss_signature
)

print("""
Bleichenbacher (1998): hücumçu seçdiyi ciphertext-lər üçün serverin RSAES-PKCS1-v1_5
encoding-i qəbul edib-etmədiyini ayırd edə bilən oracle-dan istifadə edir. Adaptiv
sorğular intervalı daraldaraq hədəf ciphertext-in plaintext-ini bərpa edir; private
exponent-i çıxarmaq hücumun nəticəsi deyil. Müdafiə yalnız xəta mətnini gizlətməkdən
ibarət deyil: vahid davranış/timing, protokol səviyyəli tədbirlər və uyğun olduqda
RSAES-OAEP kimi müasir sxem tələb olunur.

Randomization: RSAES-PKCS1-v1_5 şifrləməsi də random padding string istifadə edir;
RSAES-OAEP seed ilə randomlaşdırılır; RSASSA-PSS random salt istifadə edir;
RSASSA-PKCS1-v1_5 imzası isə deterministik encoding-dir.
""")

## 🔐 PyCryptodome ilə tam RSA tətbiqi (15 dəq)

In [ ]:
def rsa_complete_demo():
    """Açar generasiyası + RSAES-OAEP + RSASSA-PSS üçün yoxlanılan axın."""
    key = RSA.generate(2048)
    public_key = key.publickey()
    message = "RSA sxemlərini ayırırıq!".encode("utf-8")

    ciphertext = PKCS1_OAEP.new(public_key, hashAlgo=SHA256).encrypt(message)
    decrypted = PKCS1_OAEP.new(key, hashAlgo=SHA256).decrypt(ciphertext)
    assert decrypted == message

    signature = pss.new(key).sign(SHA256.new(message))
    pss.new(public_key).verify(SHA256.new(message), signature)
    assert len(ciphertext) == key.size_in_bytes()
    assert len(signature) == key.size_in_bytes()

    print("✅ Tam axın: RSAES-OAEP round-trip və RSASSA-PSS verify keçdi")
    return key, ciphertext, signature

complete_key, complete_ciphertext, complete_signature = rsa_complete_demo()

### ✍️ Çalışma 5: Tam RSA (1 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **3072-bit RSA:** PyCryptodome ilə 3072-bit RSA açar cütü yaradın və bit ölçüsünü assert edin.

2. **Uzun mesaj:** uzun məlumatı AES-GCM ilə şifrləyin, yalnız təsadüfi AES açarını RSAES-OAEP ilə sarıyın; round-trip və tamper testləri yazın.

3. **Fayl imzası:** faylın baytlarını RSASSA-PSS ilə imzalayın və dəyişdirilmiş məzmunun rəddini yoxlayın.

4. **PEM:** private key-i passphrase ilə PKCS #8 formatında export/import edin; açıq açarı ayrıca export edin. Real passphrase-i notebook-a yazmayın.

In [ ]:
def hybrid_encrypt(public_key, plaintext):
    aes_key = get_random_bytes(32)
    aes_cipher = AES.new(aes_key, AES.MODE_GCM)
    ciphertext, tag = aes_cipher.encrypt_and_digest(plaintext)
    wrapped_key = PKCS1_OAEP.new(public_key, hashAlgo=SHA256).encrypt(aes_key)
    return wrapped_key, aes_cipher.nonce, ciphertext, tag

def hybrid_decrypt(private_key, package):
    wrapped_key, nonce, ciphertext, tag = package
    aes_key = PKCS1_OAEP.new(private_key, hashAlgo=SHA256).decrypt(wrapped_key)
    return AES.new(aes_key, AES.MODE_GCM, nonce=nonce).decrypt_and_verify(ciphertext, tag)

start = time.perf_counter()
key3072 = RSA.generate(3072)
elapsed_ms = (time.perf_counter() - start) * 1000
assert key3072.n.bit_length() == 3072

long_message = ("Uzun məlumat bloku — ".encode("utf-8")) * 400
hybrid_package = hybrid_encrypt(key3072.publickey(), long_message)
assert hybrid_decrypt(key3072, hybrid_package) == long_message

file_bytes = b"example file content\n"
file_signature = pss.new(key3072).sign(SHA256.new(file_bytes))
pss.new(key3072.publickey()).verify(SHA256.new(file_bytes), file_signature)

demo_passphrase = b"classroom-only-passphrase"
private_pem = key3072.export_key(
    format="PEM", passphrase=demo_passphrase, pkcs=8,
    protection="scryptAndAES128-CBC"
)
restored_key = RSA.import_key(private_pem, passphrase=demo_passphrase)
assert restored_key.n == key3072.n and restored_key.d == key3072.d
public_pem = key3072.publickey().export_key(format="PEM")

print(f"✅ 3072-bit key ({elapsed_ms:.2f} ms), hybrid {len(long_message)} bayt, PSS və şifrəli PKCS #8 keçdi")
print(f"📦 PEM ölçüləri: private={len(private_pem)}, public={len(public_pem)} bayt")

## 🖥️ İnteqrasiya edilmiş özünü-yoxlama (15 dəq)

In [ ]:
def rsa_lab_self_test():
    """Notebook-un əsas nəticələrini interaktiv input olmadan birləşdirir."""
    assert textbook_rsa_roundtrip(42)[1] == 42
    assert textbook_byte_blocks_demo("RSA")[1] == "RSA"
    assert small_exponent_attack_demo() == 20
    assert hastad_broadcast_attack_demo() == 42
    assert hybrid_decrypt(key3072, hybrid_package) == long_message
    pss.new(complete_key.publickey()).verify(
        SHA256.new("RSA sxemlərini ayırırıq!".encode("utf-8")),
        complete_signature,
    )
    return True

assert rsa_lab_self_test()
print("✅ İnteqrasiya edilmiş self-test tamamlandı")

## 🏠 Ev tapşırığı

### 📦 Ev tapşırığı 1: RSA tədris paketi (3 bal)

Aşağıdakı modulları birləşdirən paket yazın:

```text
rsa_package/
├── __init__.py
├── textbook.py     # yalnız kiçik ədədlərlə riyazi model
├── oaep.py         # kitabxana RSAES-OAEP wrapper-i
├── pss.py          # kitabxana RSASSA-PSS wrapper-i
├── hybrid.py       # RSAES-OAEP + AES-GCM
├── attacks.py      # birbaşa kök və Håstad demosu
└── tests/          # positive və negative testlər
```

**Tələblər:** textbook modulunu production üçün olmadığını açıq yazın; private key-i
log etməyin; input domenlərini yoxlayın; hər modul üçün assertions/testlər əlavə edin;
dəyişdirilmiş ciphertext, tag, message və signature üçün mənfi testlər qurun.

### 🔐 Ev tapşırığı 2: Praktik məsələlər (2 bal)

Aşağıdakı məsələləri həll edin:

1. **Açar generasiyası:** 2048-bit RSA açar cütü yaradın. Private key-i passphrase ilə şifrələnmiş PKCS #8 PEM, public key-i ayrıca PEM kimi yazın.

2. **Hibrid şifrləmə:** faylı AES-GCM ilə şifrləyin, AES açarını RSAES-OAEP ilə sarıyın; dəyişdirilmiş tag/ciphertext-in rəddini test edin.

3. **PSS imza:** RSASSA-PSS ilə bir faylı imzalayın və həm düzgün, həm dəyişdirilmiş faylda yoxlayın.

4. **Əl ilə hesablama:** $p=101$, $q=103$, $e=7$ üçün RSA açar generasiyası aparın. Əvvəl $gcd(e,\phi(n))=1$ və $0\le m=1234<n$ şərtlərini yoxlayın, sonra şifrləyib deşifrləyin.

### 📚 Ev tapşırığı 3: Tədqiqat (2 bal)

Araşdırma aparın və cavablarınızı 1–2 səhifəlik hesabat şəklində təqdim edin:

1. **RSA-nın müəllifləri:** Rivest, Shamir və Adlemanın 1978-ci il məqaləsində public-key encryption və signature ideyaları necə təqdim olunur?

2. **Bleichenbacher (1998):** oracle hansı informasiyanı sızdırır və hücum plaintext-i necə bərpa edir? Private key bərpa olunurmu?

3. **Wiener (1990):** davamlı kəsrlər üsulu hansı əlavə fərziyyələrlə kiçik $d$-ni tapır?

4. **Aktual ölçü seçimi:** hesabat tarixini göstərin və NIST SP 800-57 Part 1 Rev. 5-də RSA-2048/RSA-3072 üçün verilən təxmini classical security strength-ləri müqayisə edin. “Hansı ildə sındırılacaq?” kimi proqnoz verməyin; tətbiqin tələb etdiyi strength/cryptoperiod-u əsaslandırın.

**Format:** PDF, ən azı 5 mənbə; RFC/NIST/orijinal məqalələrə üstünlük verin, sitat və parafrazı ayırın.

## 📌 Yekun və müzakirə sualları

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; border-left: 5px solid #2c3e50;">
<h3>📋 Xülasə</h3>
<ul>
    <li>✅ <b>Textbook RSA:</b> yalnız $0\le m&lt;n$ tam ədədləri üçün riyazi modeldir; real mesaj sxemi deyil.</li>
    <li>✅ <b>Açar invariantləri:</b> $p\ne q$, $\gcd(e,\phi(n))=1$ və $ed\equiv1\pmod{\phi(n)}$ yoxlanılır.</li>
    <li>✅ <b>Kiçik eksponent:</b> problem $e=3$ etiketindən çox, randomized encoding olmadan birbaşa kök/broadcast şərtləridir.</li>
    <li>✅ <b>Şifrləmə:</b> RSAES-OAEP; uzun məlumat üçün RSAES-OAEP + AEAD hibridi.</li>
    <li>✅ <b>İmza:</b> RSASSA-PSS; encryption və signature təhlükəsizlik anlayışları qarışdırılmır.</li>
    <li>✅ <b>Ölçü:</b> NIST SP 800-57 Rev. 5 RSA-2048-i təxminən 112-bit, RSA-3072-ni təxminən 128-bit classical strength ilə uyğunlaşdırır; seçim tətbiqin tələbinə bağlıdır.</li>
</ul>
</div>

### 💭 Müzakirə sualları

1. Textbook RSA niyə birbaşa mətn şifrləmə API-si deyil?
2. Birbaşa kub kök və Håstad hücumlarının ayrı-ayrı şərtləri hansılardır?
3. RSAES-OAEP ilə RSASSA-PSS hansı müxtəlif əməliyyatları yerinə yetirir?
4. Bleichenbacher oracle private key deyil, hansı nəticəni bərpa edir?
5. Uzun fayl üçün hibrid RSA+AEAD dizaynında hansı komponent nəyi şifrləyir?

## 📚 Əlavə resurslar

* **RFC 8017 — PKCS #1 v2.2:** [RSAES-OAEP, RSAES-PKCS1-v1_5, RSASSA-PSS və RSASSA-PKCS1-v1_5](https://www.rfc-editor.org/info/rfc8017/)
* **NIST SP 800-57 Part 1 Rev. 5:** [security strength və açar ölçüləri](https://csrc.nist.gov/pubs/sp/800/57/pt1/r5/final)
* **Rivest, Shamir, Adleman (1978):** [A Method for Obtaining Digital Signatures and Public-Key Cryptosystems](https://people.csail.mit.edu/rivest/pubs/pubs/RSA78.pdf)
* **Wiener (1990):** *Cryptanalysis of Short RSA Secret Exponents*
* **Bleichenbacher (1998):** *Chosen Ciphertext Attacks Against Protocols Based on the RSA Encryption Standard PKCS #1*
* **Boneh (1999):** [Twenty Years of Attacks on the RSA Cryptosystem](https://crypto.stanford.edu/~dabo/abstracts/RSAattack-survey.html)
* **PyCryptodome:** [RSA](https://pycryptodome.readthedocs.io/en/latest/src/public_key/rsa.html), [OAEP](https://pycryptodome.readthedocs.io/en/latest/src/cipher/oaep.html), [PSS](https://pycryptodome.readthedocs.io/en/latest/src/signature/pkcs1_pss.html)

---

<div style="background-color: #f0f0f0; padding: 20px; border-radius: 10px; text-align: center;">
<h2>✅ Məşğələ 19 tamamlandı!</h2>
<p>Notebook-un bütün code cells-i yuxarıdan aşağı assertions ilə işləməlidir.</p>
</div>